In [1]:
# Import basic packgaes 
import numpy as np
import pandas as pd
import os
from scipy.signal import argrelextrema

# Import plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from sklearn.preprocessing import MinMaxScaler
from scipy.interpolate import interp1d

# Set Color Palettes
palette1 = itertools.cycle(sns.color_palette(palette='Set1'))
palette2 = itertools.cycle(sns.color_palette(palette='Set2'))

In [2]:
directory_name = "D:\\New folder"

In [3]:
os.chdir(directory_name)

In [4]:
os.getcwd()

'D:\\New folder'

In [5]:
# Add csv files in the directory to a list
csvs = [x for x in os.listdir(directory_name) if x.endswith('.csv')]

# Add filenames of csv files to a list
fns = [os.path.splitext(os.path.basename(x))[0] for x in csvs]

# Create an empty dictionary
dic_csv = {}

# Load csv files into the dictionary
for i in range(len(fns)):
    dic_csv[fns[i]] = pd.read_csv(csvs[i])

In [6]:
# Specify the column you want to extract
column_name = 'Amplitude'

# Create a new DataFrame to store the specified column from each DataFrame
result_df = pd.DataFrame()

# Loop through each key and DataFrame in dic_csv
for key, df in dic_csv.items():
    # Extract the specified column and add it to the result DataFrame
    result_df[key] = df[column_name]


KeyError: 'Amplitude'

In [ ]:
result_df.rename({result_df.columns[0]:'Dfo',
                  result_df.columns[1]:'Sf2'},axis=1,inplace=True)


In [ ]:
#Create New Data frame to store the max values
Max_df = pd.DataFrame()

# Define the time interval for slicing
start_time = result_df.index[0]
end_time = result_df.index[-1]
time_interval = 60

In [ ]:
for i in result_df.columns:
    max_points = []
    current_time = start_time
    while current_time <= end_time:
                
        next_time = current_time + time_interval

        sliced_df = pd.DataFrame()  

        # Slice the DataFrame based on the specified time range
        sliced_df = result_df.loc[(result_df.index >= current_time) & (result_df.index <= next_time)]  

        # Perform FFT analysis only if there is data in the sliced DataFrame
        n = len(sliced_df)
        if not sliced_df.empty:
            # Append the DataFrame to the list
            max_row = sliced_df.loc[sliced_df[i].idxmax()]
            max_points.append(max_row)
                            
        current_time = next_time
    max_points_df = pd.DataFrame(max_points)
    Max_df[i] = max_points_df[i].values

In [ ]:
result_df['Dfo']

In [ ]:
diaphragm_hr = [n for n in np.arange(0,280,(280/len(Max_df)))]

Max_df['Diaphragm_life'] = diaphragm_hr

Max_df.set_index(['Diaphragm_life'], inplace=True)

In [ ]:
len(Max_df)

In [ ]:
Max_df.to_csv(r"D:\\Df0,Sf2\\T17_20_Short-test_Df0,Sf2_Without Norm.csv")

In [ ]:
a = len(Max_df.columns)  # number of rows
b = 1  # number of columns
c = 1  # initialize plot counter
fig1 = plt.figure(figsize=(25,10))

SMALL_SIZE = 25
    
for i in Max_df.columns:
    # Create a subplot and save the plot
    plt.subplot(a, b, c)
    plt.plot(Max_df.index, Max_df[i],marker = '.')
    plt.xticks(np.arange(np.min(Max_df.index),np.max((Max_df.index)),100))             
    #plt.ylim(80,110)
    #plt.title("Df0 vs Diaphragm Life Time")
    #plt.legend(df.columns, loc='lower center', bbox_to_anchor=(0.5, -0.14), ncol=8)
    c = c + 1

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=SMALL_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
# Save the plot with a filename indicating the time interval

In [ ]:

# perform a robust scaler transform of the dataset
minmax_1c = MinMaxScaler()
minmax_sf_1c = minmax_1c.fit_transform(Max_df)
Norm_df = pd.DataFrame(minmax_sf_1c, columns=['Df0','Sf2'])

In [ ]:
diaphragm_hr = [n for n in np.arange(0,280,(280/len(Norm_df)))]

Norm_df['Diaphragm_life'] = diaphragm_hr

Norm_df.set_index(['Diaphragm_life'], inplace=True)

In [ ]:
a = len(Norm_df.columns)  # number of rows
b = 1  # number of columns
c = 1  # initialize plot counter
fig1 = plt.figure(figsize=(25,15))

SMALL_SIZE = 25
    
for i in Norm_df.columns:
    # Create a subplot and save the plot
    plt.subplot(a, b, c)
    plt.plot(Norm_df.index, Norm_df[i],marker = '.')
    plt.xticks(np.arange(np.min(Norm_df.index),np.max((Norm_df.index)),10))             
    #plt.ylim(80,110)
    #plt.title(f"{i}{'vs Diaphragm Life Time'}")
    #plt.legend(df.columns, loc='lower center', bbox_to_anchor=(0.5, -0.14), ncol=8)
    c = c + 1

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=SMALL_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
# Save the plot with a filename indicating the time interval

In [ ]:
f = interp1d(x=Norm_df.index,y=Norm_df['Df0'],kind=2)
x2 = np.linspace(start=0,stop=278.05,num=10000)
y2 = f(x2)

In [ ]:
plt.figure(figsize=(25,5))
plt.plot(x2, y2, color='b')
plt.plot(Norm_df.index,Norm_df['Df0'], ls='', marker='o', color='r')
plt.xlabel('Hrs')
plt.show()

In [ ]:
f1 = interp1d(x=Norm_df.index,y=Norm_df['Sf2'],kind=2)
x3 = np.linspace(start=0,stop=278.05,num=10000)
y3 = f1(x3)

In [ ]:
plt.figure(figsize=(25,5))
plt.plot(x3, y3, color='b')
plt.plot(Norm_df.index,Norm_df['Sf2'], ls='', marker='o', color='r')
plt.xlabel('Hrs')
plt.show()

In [ ]:
Norm_df.to_csv(r"D:\\Df0,Sf2\\T04_S4_15_280Hrs_Df0,Sf2.csv")